**A. Create (Building the Catalog)**

In [12]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]
koleksi_menu.delete_many({}) # Bersihkan dulu data yang lama jika ada

koleksi_menu.insert_many([
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4},
    {"nama_menu": "Latte", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5},
    {"nama_menu": "Mocha", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 3},
    {"nama_menu": "Macchiato", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4},
    {"nama_menu": "Nasi Goreng Spesial", "harga": 30000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Mie Goreng Jawa", "harga": 25000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Ayam Geprek", "harga": 28000, "kategori": "Makanan", "pedas_level": 3},
    {"nama_menu": "Spaghetti Bolognese", "harga": 35000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Pisang Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Es Teh Manis", "harga": 10000, "kategori": "Non-Kopi"},
    {"nama_menu": "Matcha Latte", "harga": 27000, "kategori": "Non-Kopi"}
])
print("12 Menu berhasil dimasukkan ke katalog. Total dokumen:", koleksi_menu.count_documents({}))

12 Menu berhasil dimasukkan ke katalog. Total dokumen: 12


**B. Read (Multi-Level Queries)**

In [13]:
print("--- 1. Menu Kopi dengan tingkat_kepahitan > 3 ---")
for p in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {p['nama_menu']} (Kepahitan: {p['tingkat_kepahitan']})")

print("\n--- 2. Menu dengan harga 15.000 - 30.000 ---")
for p in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']})")

print("\n--- 3. Menu tanpa field pedas_level ---")
for p in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {p['nama_menu']} (Kategori: {p['kategori']})")

--- 1. Menu Kopi dengan tingkat_kepahitan > 3 ---
  - Americano (Kepahitan: 4)
  - Espresso (Kepahitan: 5)
  - Macchiato (Kepahitan: 4)

--- 2. Menu dengan harga 15.000 - 30.000 ---
  - Americano (Rp20000)
  - Latte (Rp25000)
  - Espresso (Rp18000)
  - Mocha (Rp28000)
  - Macchiato (Rp22000)
  - Nasi Goreng Spesial (Rp30000)
  - Mie Goreng Jawa (Rp25000)
  - Ayam Geprek (Rp28000)
  - Pisang Bakar (Rp15000)
  - Matcha Latte (Rp27000)

--- 3. Menu tanpa field pedas_level ---
  - Americano (Kategori: Kopi)
  - Latte (Kategori: Kopi)
  - Espresso (Kategori: Kopi)
  - Mocha (Kategori: Kopi)
  - Macchiato (Kategori: Kopi)
  - Es Teh Manis (Kategori: Non-Kopi)
  - Matcha Latte (Kategori: Non-Kopi)


**C. Update (Status Overhaul)**

In [14]:
# Tambah field status "Tersedia" ke semua dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah 2 menu menjadi "Habis"
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek"}, {"$set": {"status": "Habis"}})

print("--- Status Produk Habis ---")
for p in koleksi_menu.find({"status": "Habis"}):
    print(f"  - {p['nama_menu']}")

--- Status Produk Habis ---
  - Espresso
  - Ayam Geprek


**D: Spark Integration**

In [15]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import avg

spark = SparkSession.builder \
    .appName("Tugas9_Dany_Akhdan_Imaduddin") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Ambil data lalu convert ke pandas
dokumen = list(koleksi_menu.find({}))
df_pd = pd.DataFrame(dokumen)
df_pd["_id"] = df_pd["_id"].astype(str) # Ubah ObjectID jadi string

# Konversi ke Spark DataFrame
df_spark = spark.createDataFrame(df_pd)

print("--- 1. Print Schema ---")
df_spark.printSchema()

print("\n--- 2. Rata-rata Harga per Kategori ---")
df_spark.groupBy("kategori").agg(avg("harga").alias("rata_rata_harga")).show()

--- 1. Print Schema ---
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)


--- 2. Rata-rata Harga per Kategori ---
+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|        22600.0|
| Makanan|        26600.0|
|Non-Kopi|        18500.0|
+--------+---------------+



In [16]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


**E. Kesimpulan**

**Praktikum kali ini memberikan pemahaman mendalam mengenai bagaimana cara mengintegrasikan basis data NoSQL seperti MongoDB dengan sistem pemrosesan terdistribusi Apache Spark. Melalui eksplorasi instalasi server, pengoperasian perintah CRUD dasar menggunakan PyMongo, hingga teknik konversi data semi-terstruktur ke dalam format tabel Spark DataFrame, kita dapat melihat bagaimana kedua teknologi ini saling melengkapi. MongoDB unggul dalam menangani data yang bentuk dan atributnya tidak seragam antar dokumen, sementara Spark sangat diandalkan untuk melakukan analisis data skala besar secara cepat. Penggunaan Pandas DataFrame sebagai jembatan perantara terbukti sangat efektif untuk meratakan perbedaan struktur data dan mengatasi masalah kolom kosong sebelum data tersebut diolah lebih lanjut menggunakan engine Spark. Dengan demikian, penguasaan alur kerja end-to-end ini menjadi bekal yang sangat berharga dalam membangun arsitektur pengolahan data modern yang fleksibel, tangguh, dan efisien untuk berbagai skenario di dunia nyata.**